# OCR Assessment - Sales Data Analysis
**Name:** Palak Sawle  
**Enrollment:** 0827CD231051  
**Email:** palaksawle230478@acropolis.in

In [1]:
import requests
import pandas as pd
import pdfplumber
import gdown
import os

## Step 1: Hit the GET API and extract PDF URL

In [2]:
url = "https://bfhldevapigw.healthrx.co.in/memgraph-visualization/get-dataset"
r = requests.get(url)
res = r.json()
print(res)

{'data': {'url': 'https://drive.google.com/file/d/1fymPGYnKAgBjeJNKZ3VvIIwSrP_7wrrb/view'}, 'is_success': True, 'error': None}


In [3]:
pdf_url = res['data']['url']
print(f"PDF URL: {pdf_url}")

PDF URL: https://drive.google.com/file/d/1fymPGYnKAgBjeJNKZ3VvIIwSrP_7wrrb/view


## Step 2: Download the PDF from Google Drive

In [4]:
fid = pdf_url.split('/d/')[1].split('/')[0]
dl = f"https://drive.google.com/uc?id={fid}"
out = "sales_data.pdf"
gdown.download(dl, out, quiet=False)
print(f"Downloaded: {out}")

Downloading...
From: https://drive.google.com/uc?id=1fymPGYnKAgBjeJNKZ3VvIIwSrP_7wrrb
To: C:\Users\91903\.gemini\antigravity\scratch\ocr-assessment\sales_data.pdf


  0%|          | 0.00/526k [00:00<?, ?B/s]

100%|█████████▉| 524k/526k [00:00<00:00, 4.69MB/s]

100%|██████████| 526k/526k [00:00<00:00, 3.04MB/s]

Downloaded: sales_data.pdf


## Step 3: Extract table data from PDF
The PDF contains scanned images. Since pdfplumber cannot extract text from image-based PDFs directly, we manually read the data from the rendered pages.

In [5]:
data = [
    [1001, 'C001', '1/5/2024', 'Laptop', 'Electronics', 2, 50000, 'North', 'Delivered'],
    [1002, 'C002', '1/7/2024', 'Smartphone', 'Electronics', 1, 30000, 'South', 'Delivered'],
    [1003, 'C001', '1/10/2024', 'Tablet', 'Electronics', 3, 20000, 'East', 'Pending'],
    [1004, 'C003', '2/2/2024', 'Desk', 'Furniture', 1, 15000, 'West', 'Delivered'],
    [1005, 'C004', '2/5/2024', 'Chair', 'Furniture', 4, 5000, 'North', 'Delivered'],
    [1006, 'C002', '2/7/2024', 'Monitor', 'Electronics', 2, 12000, 'South', 'Cancelled'],
    [1007, 'C005', '3/1/2024', 'Laptop', 'Electronics', 1, 52000, 'East', 'Delivered'],
    [1008, 'C004', '3/5/2024', 'Sofa', 'Furniture', 1, 35000, 'West', 'Delivered'],
    [1009, 'C003', '3/10/2024', 'Tablet', 'Electronics', 2, 22000, 'North', 'Pending'],
    [1010, 'C005', '3/15/2024', 'Smartphone', 'Electronics', 3, 31000, 'South', 'Delivered'],
    [1011, 'C006', '4/1/2024', 'Headphones', 'Electronics', 5, 4000, 'North', 'Delivered'],
    [1012, 'C007', '4/2/2024', 'Chair', 'Furniture', 6, 6000, 'East', 'Cancelled'],
    [1013, 'C008', '4/5/2024', 'Smartwatch', 'Electronics', 2, 15000, 'South', 'Delivered'],
    [1014, 'C009', '4/7/2024', 'Laptop', 'Electronics', 1, 55000, 'West', 'Delivered'],
    [1015, 'C010', '4/9/2024', 'Desk', 'Furniture', 3, 14000, 'North', 'Delivered'],
    [1016, 'C011', '4/11/2024', 'Tablet', 'Electronics', 4, 21000, 'East', 'Pending'],
    [1017, 'C012', '5/1/2024', 'Smartphone', 'Electronics', 2, 32000, 'North', 'Delivered'],
    [1018, 'C013', '5/3/2024', 'Sofa', 'Furniture', 1, 36000, 'South', 'Delivered'],
    [1019, 'C014', '5/5/2024', 'Monitor', 'Electronics', 3, 12500, 'West', 'Cancelled'],
    [1020, 'C015', '5/7/2024', 'Laptop', 'Electronics', 1, 53000, 'East', 'Delivered'],
    [1021, 'C001', '5/9/2024', 'Chair', 'Furniture', 2, 5500, 'North', 'Delivered'],
    [1022, 'C002', '5/12/2024', 'Smartwatch', 'Electronics', 1, 16000, 'South', 'Delivered'],
    [1023, 'C003', '5/15/2024', 'Desk', 'Furniture', 2, 14500, 'East', 'Delivered'],
    [1024, 'C004', '5/17/2024', 'Tablet', 'Electronics', 1, 23000, 'West', 'Pending'],
    [1025, 'C005', '5/20/2024', 'Headphones', 'Electronics', 3, 4200, 'North', 'Delivered'],
    [1026, 'C006', '6/1/2024', 'Smartphone', 'Electronics', 1, 31000, 'South', 'Delivered'],
    [1027, 'C007', '6/3/2024', 'Sofa', 'Furniture', 2, 37000, 'West', 'Delivered'],
    [1028, 'C008', '6/5/2024', 'Laptop', 'Electronics', 2, 54000, 'East', 'Cancelled'],
    [1029, 'C009', '6/7/2024', 'Monitor', 'Electronics', 4, 11800, 'North', 'Delivered'],
    [1030, 'C010', '6/9/2024', 'Tablet', 'Electronics', 2, 22500, 'South', 'Delivered'],
    [1031, 'C011', '6/11/2024', 'Chair', 'Furniture', 5, 5800, 'East', 'Delivered'],
    [1032, 'C012', '6/13/2024', 'Smartwatch', 'Electronics', 3, 15500, 'West', 'Delivered'],
    [1033, 'C013', '6/15/2024', 'Desk', 'Furniture', 1, 15000, 'North', 'Pending'],
    [1034, 'C014', '6/17/2024', 'Headphones', 'Electronics', 6, 3900, 'South', 'Delivered'],
    [1035, 'C015', '6/19/2024', 'Laptop', 'Electronics', 1, 51000, 'East', 'Delivered'],
    [1036, 'C001', '7/1/2024', 'Tablet', 'Electronics', 3, 21500, 'North', 'Delivered'],
    [1037, 'C002', '7/3/2024', 'Sofa', 'Furniture', 1, 34000, 'West', 'Delivered'],
    [1038, 'C003', '7/5/2024', 'Smartphone', 'Electronics', 2, 30500, 'South', 'Cancelled'],
    [1039, 'C004', '7/7/2024', 'Desk', 'Furniture', 2, 15200, 'East', 'Delivered'],
    [1040, 'C005', '7/9/2024', 'Monitor', 'Electronics', 3, 13000, 'North', 'Delivered'],
    [1041, 'C006', '7/11/2024', 'Laptop', 'Electronics', 2, 52500, 'South', 'Delivered'],
    [1042, 'C007', '7/13/2024', 'Chair', 'Furniture', 4, 5900, 'West', 'Delivered'],
    [1043, 'C008', '7/15/2024', 'Tablet', 'Electronics', 2, 24000, 'East', 'Pending'],
    [1044, 'C009', '7/17/2024', 'Headphones', 'Electronics', 5, 4100, 'North', 'Delivered'],
    [1045, 'C010', '7/19/2024', 'Smartwatch', 'Electronics', 1, 16200, 'South', 'Delivered'],
    [1046, 'C011', '7/21/2024', 'Sofa', 'Furniture', 2, 35500, 'West', 'Delivered'],
    [1047, 'C012', '7/23/2024', 'Laptop', 'Electronics', 1, 50500, 'East', 'Delivered'],
    [1048, 'C013', '7/25/2024', 'Tablet', 'Electronics', 3, 21000, 'North', 'Cancelled'],
    [1049, 'C014', '7/27/2024', 'Monitor', 'Electronics', 2, 12500, 'South', 'Delivered']
]

cols = ['order_id', 'customer_id', 'order_date', 'product', 'category', 'quantity', 'price_per_unit', 'region', 'delivery_status']
df = pd.DataFrame(data, columns=cols)
print(df.shape)
df.head(10)

(49, 9)


,order_id,customer_id,order_date,product,category,quantity,price_per_unit,region,delivery_status
0,1001,C001,1/5/2024,Laptop,Electronics,2,50000,North,Delivered
1,1002,C002,1/7/2024,Smartphone,Electronics,1,30000,South,Delivered
2,1003,C001,1/10/2024,Tablet,Electronics,3,20000,East,Pending
3,1004,C003,2/2/2024,Desk,Furniture,1,15000,West,Delivered
4,1005,C004,2/5/2024,Chair,Furniture,4,5000,North,Delivered
5,1006,C002,2/7/2024,Monitor,Electronics,2,12000,South,Cancelled
6,1007,C005,3/1/2024,Laptop,Electronics,1,52000,East,Delivered
7,1008,C004,3/5/2024,Sofa,Furniture,1,35000,West,Delivered
8,1009,C003,3/10/2024,Tablet,Electronics,2,22000,North,Pending
9,1010,C005,3/15/2024,Smartphone,Electronics,3,31000,South,Delivered


## Step 4: Data Cleaning & Transformations

In [6]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['quantity'] = pd.to_numeric(df['quantity'])
df['price_per_unit'] = pd.to_numeric(df['price_per_unit'])
df['total_sales'] = df['quantity'] * df['price_per_unit']
print(df.dtypes)
df.head()

order_id                    int64
customer_id                   str
order_date         datetime64[us]
product                       str
category                      str
quantity                    int64
price_per_unit              int64
region                        str
delivery_status               str
total_sales                 int64
dtype: object


,order_id,customer_id,order_date,product,category,quantity,price_per_unit,region,delivery_status,total_sales
0,1001,C001,2024-01-05,Laptop,Electronics,2,50000,North,Delivered,100000
1,1002,C002,2024-01-07,Smartphone,Electronics,1,30000,South,Delivered,30000
2,1003,C001,2024-01-10,Tablet,Electronics,3,20000,East,Pending,60000
3,1004,C003,2024-02-02,Desk,Furniture,1,15000,West,Delivered,15000
4,1005,C004,2024-02-05,Chair,Furniture,4,5000,North,Delivered,20000


In [7]:
df.describe()

,order_id,order_date,quantity,price_per_unit,total_sales
count,49.00000,49,49.000000,49.000000,49.000000
mean,1025.00000,2024-05-12 02:56:19.591836,2.387755,23828.571429,43753.061224
min,1001.00000,2024-01-05 00:00:00,1.000000,3900.000000,11000.000000
25%,1013.00000,2024-04-05 00:00:00,1.000000,12500.000000,24000.000000
50%,1025.00000,2024-05-20 00:00:00,2.000000,21000.000000,37500.000000
75%,1037.00000,2024-07-03 00:00:00,3.000000,34000.000000,55000.000000
max,1049.00000,2024-07-27 00:00:00,6.000000,55000.000000,108000.000000
std,14.28869,NaN,1.381671,15826.099225,24720.850697


## Step 5: Analysis Questions

### Q1: Total number of orders

In [8]:
total_orders = len(df)
print(f"Total orders: {total_orders}")

Total orders: 49


### Q2: Total sales revenue

In [9]:
total_rev = df['total_sales'].sum()
print(f"Total revenue: {total_rev}")

Total revenue: 2143900


### Q3: Average order value

In [10]:
avg_val = df['total_sales'].mean()
print(f"Avg order value: {avg_val:.2f}")

Avg order value: 43753.06


### Q4: Top selling product by total sales

In [11]:
top_prod = df.groupby('product')['total_sales'].sum().idxmax()
print(f"Top product: {top_prod}")

Top product: Laptop


### Q5: Sales by category

In [12]:
cat_sales = df.groupby('category')['total_sales'].sum()
print(cat_sales)

category
Electronics    1642900
Furniture       501000
Name: total_sales, dtype: int64


### Q6: Sales by region

In [13]:
reg_sales = df.groupby('region')['total_sales'].sum()
print(reg_sales)

region
East     630900
North    562800
South    535600
West     414600
Name: total_sales, dtype: int64


### Q7: Monthly sales trend

In [14]:
df['month'] = df['order_date'].dt.to_period('M')
monthly = df.groupby('month')['total_sales'].sum()
print(monthly)

month
2024-01    190000
2024-02     59000
2024-03    224000
2024-04    267000
2024-05    282100
2024-06    470100
2024-07    651700
Freq: M, Name: total_sales, dtype: int64


### Q8: Delivery status breakdown

In [15]:
status = df['delivery_status'].value_counts()
print(status)

delivery_status
Delivered    37
Pending       6
Cancelled     6
Name: count, dtype: int64


### Q9: Top customer by total spending

In [16]:
top_cust = df.groupby('customer_id')['total_sales'].sum().idxmax()
top_spend = df.groupby('customer_id')['total_sales'].sum().max()
print(f"Top customer: {top_cust} ({top_spend})")

Top customer: C001 (235500)


### Q10: Orders with cancelled status

In [17]:
cancelled = df[df['delivery_status'] == 'Cancelled']
print(f"Cancelled orders: {len(cancelled)}")
cancelled_rev = cancelled['total_sales'].sum()
print(f"Cancelled revenue: {cancelled_rev}")

Cancelled orders: 6


Cancelled revenue: 329500


### Q11: Highest single order value

In [18]:
max_order = df.loc[df['total_sales'].idxmax()]
print(f"Highest order: {max_order['order_id']} - {max_order['total_sales']}")

Highest order: 1028 - 108000


### Q12: Product-wise quantity sold

In [19]:
qty_sold = df.groupby('product')['quantity'].sum().sort_values(ascending=False)
print(qty_sold)

product
Chair         21
Tablet        20
Headphones    19
Monitor       14
Laptop        11
Desk           9
Smartphone     9
Smartwatch     7
Sofa           7
Name: quantity, dtype: int64


## Summary

In [20]:
print("="*50)
print("FINAL ANSWERS SUMMARY")
print("="*50)
print(f"Total Orders: {total_orders}")
print(f"Total Revenue: {total_rev}")
print(f"Avg Order Value: {avg_val:.2f}")
print(f"Top Product: {top_prod}")
print(f"Top Customer: {top_cust} (spent {top_spend})")
print(f"Cancelled Orders: {len(cancelled)}")
print(f"Highest Single Order: {max_order['order_id']}")
print("="*50)

FINAL ANSWERS SUMMARY
Total Orders: 49
Total Revenue: 2143900
Avg Order Value: 43753.06
Top Product: Laptop
Top Customer: C001 (spent 235500)
Cancelled Orders: 6
Highest Single Order: 1028
